In [29]:
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm import tqdm

In [30]:
dataset = pd.read_parquet('train-00000-of-00001-1ae224438dce829b.parquet')
dataset
results_df = pd.DataFrame(columns=["instruction", "reference", "generated", "bleu_score"])

In [31]:
def load_model_and_tokenizer(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path)
    return model, tokenizer

In [32]:
def generate_response(model, tokenizer, prompt, max_length=256):
    # Tokenize the input
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Initialize the progress bar
    total_steps = max_length  # Number of decoding steps corresponds to max_length
    with tqdm(total=total_steps, desc="Generating response", unit="step") as pbar:
        # Generate the response
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=5,
            early_stopping=True,
            # Update the progress bar on each decoding step
            output_scores=True, 
            return_dict_in_generate=True,
        )
        
        # Update the progress bar
        for _ in range(len(outputs.sequences[0])):
            pbar.update(1)
    
    # Decode the generated response
    response = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
    return response

In [33]:
def evaluate_model(dataset, model, tokenizer):
    global results_df
    bleu_scores = []
    for idx in range(len(dataset)):
        instruction = dataset.at[idx, "instruction"]
        reference = dataset.at[idx, "output"] 
        generated_text = generate_response(model, tokenizer, instruction)

        bleu_score = sentence_bleu(reference, generated_text, weights=(0.25,0.25,0.25,0.25))
        bleu_scores.append(bleu_score)

        print(f"Input: {instruction}")
        print(f"Reference: {reference}")
        print(f"Generated: {generated_text}")
        print(f"BLEU Score: {bleu_score}\n")
                # DataFrame에 결과 추가
        results_df = pd.concat([results_df, pd.DataFrame({
            "instruction": [instruction],
            "reference": [reference],
            "generated": [generated_text],
            "bleu_score": [bleu_score]
        })], ignore_index=True)

    # CSV 파일로 저장
    results_df.to_csv("llama-3.2-Korean-Bllossom-3B_evaluation.csv", index=False)

In [ ]:
if __name__ == "__main__":
    parquet_path = "train-00000-of-00001-1ae224438dce829b.parquet"  
    model_path = "./models--Bllossom--llama-3.2-Korean-Bllossom-3B/snapshots/67c476d8de2afd021661a40c0e5f5cbb51e62bc6"  
#models--Bllossom--llama-3.2-Korean-Bllossom-3B
    model = AutoModelForCausalLM.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    # 평가 실행
    evaluate_model(dataset, model, tokenizer)

Some weights of LlamaForCausalLM were not initialized from the model checkpoint at ./models--Bllossom--llama-3.2-Korean-Bllossom-3B/snapshots/67c476d8de2afd021661a40c0e5f5cbb51e62bc6 and are newly initialized: ['model.layers.20.input_layernorm.weight', 'model.layers.20.mlp.down_proj.weight', 'model.layers.20.post_attention_layernorm.weight', 'model.layers.21.input_layernorm.weight', 'model.layers.21.mlp.down_proj.weight', 'model.layers.21.mlp.gate_proj.weight', 'model.layers.21.mlp.up_proj.weight', 'model.layers.21.post_attention_layernorm.weight', 'model.layers.21.self_attn.k_proj.weight', 'model.layers.21.self_attn.o_proj.weight', 'model.layers.21.self_attn.q_proj.weight', 'model.layers.21.self_attn.v_proj.weight', 'model.layers.22.input_layernorm.weight', 'model.layers.22.mlp.down_proj.weight', 'model.layers.22.mlp.gate_proj.weight', 'model.layers.22.mlp.up_proj.weight', 'model.layers.22.post_attention_layernorm.weight', 'model.layers.22.self_attn.k_proj.weight', 'model.layers.22.se

Input: 보드 게임 스피너는 $A$, $B$, $C$로 표시된 세 부분으로 나뉩니다. 스피너가 $A$에 떨어질 확률은 $\frac{1}{3}$이고, 스피너가 $B$에 떨어질 확률은 $\frac{5}{12}$입니다.  스피너가 $C$에 착륙할 확률은 얼마입니까? 답을 공통 분수로 표현하세요.
Reference: 모든 가능한 결과의 확률의 합이 1$이므로, 스피너가 $C$에 착륙할 확률을 구하려면 스피너가 $A$와 $B$에 착륙할 확률을 1$에서 빼야 합니다. 이를 방정식으로 쓸 수 있습니다: $P(C) = 1 - P(A) - P(B)$. P(A) = \frac{1}{3}$, $P(B) = \frac{5}{12}$라는 것을 알고 있으므로 이 값을 방정식에 대입하여 단순화할 수 있습니다. 결과는 다음과 같습니다: P(C) = 1 - \frac{1}{3} - frac{5}{12} = \frac{12}{12} - frac{4}{12} - frac{5}{12} = \frac{3}{12}$. 분자와 분모를 $3$로 나누면 이 분수를 줄일 수 있습니다: P(C) = \frac{1}{4}$입니다.
Generated: 보드 게임 스피너는 $A$, $B$, $C$로 표시된 세 부분으로 나뉩니다. 스피너가 $A$에 떨어질 확률은 $\frac{1}{3}$이고, 스피너가 $B$에 떨어질 확률은 $\frac{5}{12}$입니다.  스피너가 $C$에 착륙할 확률은 얼마입니까? 답을 공통 분수로 표현하세요.oderagonano escapinganoowitz Broadway Broadway Broadwayano contractingano bleibtabel sn Habit pine victims cliffs Beaveranoanche pickedanche sit cliff recordano.dp victimano schnμι直 sidides yetlanders Broadwayano publicly cliff schano contractingano cliff pin yet pocket Mi

Generating response: 100%|██████████| 256/256 [02:09<00:00,  1.98step/s] 


Input: 저희 학교 수학 클럽에는 남학생 6명과 여학생 8명이 있습니다.  주 수학 경시대회에 파견할 팀을 선발해야 합니다. 팀에 6명이 필요합니다.  제한 없이 팀을 몇 가지 방법으로 선발할 수 있나요?
Reference: 14명 중 6명을 선택해야 하는데 순서는 중요하지 않습니다. 이것은 순열 문제가 아니라 조합 문제입니다. 조합의 공식은 nCr = n! / (r! * (n-r)!)이며, 여기서 n은 총 선택의 개수이고 r은 선택의 개수입니다. 숫자를 연결하면 14C6 = 14! / (6! * 8!) = 3003.
Generated: 저희 학교 수학 클럽에는 남학생 6명과 여학생 8명이 있습니다.  주 수학 경시대회에 파견할 팀을 선발해야 합니다. 팀에 6명이 필요합니다.  제한 없이 팀을 몇 가지 방법으로 선발할 수 있나요?azzo Concerton Sit pocketsDBC victims pockets spirited pocket tallerHigher talleriet kiasa victimsamicaxy trouble Pulitzerano DEA victimsamicartoagooolsboroDBCano-pocketupo尚uyu spiritedlakeano pocketsatern kickerano-pocket rel spoiler bench pocket prey pocketano pocketano pocketsanoenary pocket prey relayano pocket publiclyano pockets rel stomachquetteano pocketsanoaca Riceano-pocket prey relayamic�edis nevertheless rel tossanoacaano pockets rel Mell curbano-pocketanoaca Pretty supervamic regul pocketanoaca rel encour chin pocketanoacaano pocketanoacaano pockets reliramanoaca re

Generating response: 100%|██████████| 256/256 [02:03<00:00,  2.07step/s] 


...

Generating response: 100%|██████████| 256/256 [02:06<00:00,  2.02step/s] 


Input: 서랍에 눈에 띄는 양말이 10개 있습니다: 흰색 4개, 갈색 4개, 파란색 2개입니다.  같은 색의 양말 두 켤레가 있다면 몇 가지 방법으로 양말 한 켤레를 선택할 수 있나요?
Reference: 같은 색의 양말 두 개를 선택하는 방법의 수를 세고 싶기 때문에 각 색을 개별적으로 고려해야 합니다. 흰색 양말의 경우 같은 양말을 반복할 수 없기 때문에 첫 번째 양말을 선택하는 방법은 4가지, 두 번째 양말을 선택하는 방법은 3가지입니다. 그러나 순서는 중요하지 않기 때문에 이것은 각 쌍을 두 번 계산합니다. 따라서 쌍 수를 얻으려면 2로 나누어야합니다. 즉, (4 * 3) / 2 = 6 쌍의 흰색 양말이 있습니다. 마찬가지로 동일한 추론으로 갈색 양말 6 켤레와 파란색 양말 1 켤레가 있습니다. 같은 색의 양말 총 쌍 수를 얻으려면이 숫자를 더해야합니다. 즉, 6 + 6 + 1 = 13쌍의 같은 색 양말이 있습니다.
Generated: 서랍에 눈에 띄는 양말이 10개 있습니다: 흰색 4개, 갈색 4개, 파란색 2개입니다.  같은 색의 양말 두 켤레가 있다면 몇 가지 방법으로 양말 한 켤레를 선택할 수 있나요?jac punct Bek synthetic cliff contin surrounding pocketeyer pocket tall tall skyslake pocketanoancheLite pocketeyer inde pickedago chaincott Authentic pocket直 anchor pocketeyerago pockets cliff contin prefix publicly Ale {{
 Expect-stat-tip-statA Bryananovalue pocketamiciramano Swampano.toJson pocket pocketanoA cliff pin pocket pocket pocketamicarel {{
agas Mỹ pocketano affair Bekagon sugar cliff contin Sugar clifffe s

Generating response: 100%|██████████| 256/256 [02:24<00:00,  1.77step/s]  


Input: 책이 선택되는 순서가 중요하지 않은 경우 6권의 책꽂이에서 4권의 책을 선택할 수 있는 방법은 몇 가지입니까?
Reference: 이 문제는 순서에 관계없이 6권 중 4권의 책 조합의 수를 세는 문제입니다. 한 번에 r을 취한 n 개 물체의 조합 수를 계산하는 공식은 nCr = n!/(r!(n-r)!)이며, 여기서 n!은 n의 계승으로 n까지의 모든 양의 정수의 곱을 의미합니다. 이 경우 n = 6, r = 4이므로 해당 값을 공식에 대입하여 단순화할 수 있습니다. 따라서 6권의 책이 있는 서가에서 순서와 상관없이 4권의 책을 선택하는 방법은 15가지가 있습니다.
Generated: 책이 선택되는 순서가 중요하지 않은 경우 6권의 책꽂이에서 4권의 책을 선택할 수 있는 방법은 몇 가지입니까? reductionsano.getDefault sit kidnappedanoheimanouga Late spiritedano Danaano Coolerupo killer/full pocket taller pocket taller pocketago� bench pocket taller Danaano pocketsano tallerano pockets pocketanoacaMajor talleritet tub pocket taller pocket Ale publiclyingle Buffalo slot Macyuples yet spirited-fluid talleritetescalanche Sheronomwaitwaitwait chin Danaano pockets rel Cadillacamyenticinteropail national taller Danaano pocket pocket religh curbano autom Danaano reliefazzo Agility pocketago pocket relwaysanoaca pocket relonomMagic-pocket relighighighigit (@ panic taller pocket reliram spirit

Generating response:   0%|          | 0/256 [00:00<?, ?step/s]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
